This notebook calculates the VaR and ES values with the Diffusion Method.



#Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
import numpy as np
import sys
sys.path.insert(0,'/content/drive/MyDrive/Masterthesis')
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy.stats import norm
import glob
import re

#Load Historical Data


In [ ]:
#This code loads the sequences from the desired file and conects them.
path_bluechip = '/content/drive/MyDrive/Masterthesis/Data/Sequences/blueChip/samples'
path_crossAsset = '/content/drive/MyDrive/Masterthesis/Data/Sequences/crossAsset/samples'
all_files = glob.glob(os.path.join(path_crossAsset, "synthetic_data_window_*.csv"))

def natural_sort_key(s):
    match = re.search(r'(\d+)\.csv$', s)
    if match:
        return int(match.group(1))
    return s
all_files.sort(key=natural_sort_key)
print(f": {all_files}")

sequence_list = []
seq_length = 10

for f in all_files:
    df = pd.read_csv(f)
    print(f"Shape of {f}: {df.shape}")
    sequence_list.append(df)

data = pd.concat(sequence_list, ignore_index=True)
sequence_per_window = 10000

In [ ]:
num_points, num_variables = data.shape
num_sequences = (num_points // seq_length)//sequence_per_window

data_reshaped = data.to_numpy().reshape(num_sequences, seq_length, sequence_per_window, num_variables)
np.shape(data_reshaped)

In [ ]:
def portfolio_returns(returns, weights=None):

    returns = np.asarray(returns, dtype=float)

    returns = np.nan_to_num(returns)

    num_sequences, seq_length_minus_1, sequence_per_window, num_variables = returns.shape
    if weights is None:
        weights = np.ones(num_variables) / num_variables
    else:
        weights = np.asarray(weights, dtype=float)
        weights = weights / np.sum(weights)


    port_ret = np.einsum('ijkl, l -> ijk', returns, weights)
    return port_ret



returns_data = portfolio_returns(data_reshaped)

In [ ]:
np.shape(returns_data)

# Historical Method


In [ ]:
#This method calculates the actuall VaR and ES values given a confidence level.

def var_es_method(returns, confidence_level=0.975, es_confidence_level=0.975, path=None):
    num_sequences, sequence_length, sequence_per_window = returns.shape

    VaR_historical = []
    ES_historical = []

    for j in range(num_sequences):
        for i in range(sequence_length):
          window_returns = returns[j, i, :].flatten()

          if len(window_returns) > 0:
              # VaR
              var = np.percentile(window_returns, (1 - confidence_level) * 100)
              VaR_historical.append(var)

              # ES
              var_for_es = np.percentile(window_returns, (1 - es_confidence_level) * 100)
              losses_worse_than_var_es = window_returns[window_returns < var_for_es]

              if len(losses_worse_than_var_es) > 0:
                  es = losses_worse_than_var_es.mean()
                  ES_historical.append(es)
              else:
                  ES_historical.append(np.nan)
          else:
              VaR_historical.append(np.nan)
              ES_historical.append(np.nan)
              continue

    VaR_array = np.array(VaR_historical)#[:-4]
    ES_array = np.array(ES_historical)#[:-4]
    #[:-4] for the last sequence since we generate 10 points per sequence
    #in our experiment we end at 756 tho.

    if path is not None:
        df_output = pd.DataFrame({
            'VaR': VaR_array,
            'ES': ES_array
        })

        df_output.to_csv(f"{path}crossAsset_Diffusion.csv", index=False)

    return VaR_array, ES_array

In [ ]:
path_bluechip = "/content/drive/MyDrive/Masterthesis/Data/Datasets/bluechip_sp500.csv"
path_crossAsset = "/content/drive/MyDrive/Masterthesis/Data/Datasets/crossAsset_portfolio.csv"
df_data = pd.read_csv(path_crossAsset)

#this method normalizes our portfolio data depending on some weights
def normal_portfolio_returns(returns, weights=None):
    returns = np.asarray(returns, dtype=float)
    returns = np.nan_to_num(returns)

    T, N = returns.shape
    if weights is None:
        weights = np.ones(N) / N
    else:
        weights = np.asarray(weights, dtype=float)
        weights = weights / np.sum(weights)

    port_ret = returns @ weights
    return port_ret

normal_returns_data = normal_portfolio_returns(df_data)


In [ ]:
crossAsset_train_length = 5298
bluechip_train_length = 4564
train_length = crossAsset_train_length

hist = var_es_method(returns_data, path='/content/drive/MyDrive/Masterthesis/Data/VaR_and_ES/')

returns = normal_returns_data
#basic plotting of the results
plot_length = len(hist[0])
plt.figure(figsize=(12, 7))
plt.plot(range(plot_length), returns[train_length:train_length +plot_length ], label='Portfolio Returns')
plt.plot(range(plot_length), hist[0], label='Historical VaR', linestyle='--')
plt.plot(range(plot_length), hist[1], label='Historical ES', linestyle=':')

exceedances_historical_var = returns[train_length:train_length+plot_length] < hist[0]
plt.plot(np.where(exceedances_historical_var)[0], returns[train_length:train_length+ plot_length][exceedances_historical_var], 'kx', markersize=7, label='Historical VaR Exceedances')
print(f"Number of Historical VaR exceedances: {np.sum(exceedances_historical_var)}")

plt.legend()
plt.grid(True, alpha=0.2)
plt.show()